<a href="https://colab.research.google.com/github/EshikaAbbaraju/Role_Specified_Collective_Foraging_Task_Model/blob/main/Role_Specified_Collective_Foraging_Task_Model_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!git clone https://github.com/JerryGuo2001/Role_Speficifed_Collective_Foraging_Task.git
#%cd Role_Speficifed_Collective_Foraging_Task

#tune the danger from aliens
#print out snapshot labels at the top

%cd /content/Role_Speficifed_Collective_Foraging_Task
CSV_DIR = "gridworld/"

Cloning into 'Role_Speficifed_Collective_Foraging_Task'...
remote: Enumerating objects: 305, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 305 (delta 21), reused 35 (delta 11), pack-reused 255 (from 1)
Receiving objects: 100% (305/305), 1.84 MiB | 7.42 MiB/s, done.
Resolving deltas: 100% (169/169), done.
/content/Role_Speficifed_Collective_Foraging_Task


In [30]:
from __future__ import annotations

"""
Collective foraging + security: agents keep positions and world state (mines, visited, etc.).
A “round” is exactly one agent’s block of `moves_per_round` steps; agents alternate each round:
round 1 -> forager moves 5x (security frozen), round 2 -> security moves 5x (forager frozen).

This version includes two behavior fixes:
1. The forager may step onto the security cell when that cell is a revealed active mine.
2. A follower forager gets an explicit bonus for safe progress toward a revealed high-value mine,
   so escort-ring / orbit incentives do not trap it in circles after the escort reveals a target.
"""

from collections import deque
from dataclasses import dataclass, replace
from typing import Dict, List, Optional, Set, Tuple

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import ListedColormap


CSV_DIR = "gridworld/" if os.path.isdir("gridworld") else "."


def _reward_observed(env: pd.DataFrame, pos: Tuple[int, int], revealed: Set[Tuple[int, int]]) -> float:
    if pos not in revealed or pos not in env.index:
        return 0.0
    return float(env.loc[pos, "reward_prob"])


def _mine_observed(env: pd.DataFrame, pos: Tuple[int, int], revealed: Set[Tuple[int, int]]) -> bool:
    if pos not in revealed or pos not in env.index:
        return False
    return str(env.loc[pos, "mine_type"]).strip() != ""


def _normalize_identity(s: Optional[str]) -> str:
    if s is None:
        return "auto"
    x = str(s).strip().lower()
    if x in ("lead", "leader", "l"):
        return "leader"
    if x in ("follow", "follower", "f"):
        return "follower"
    if x in ("auto", "a", "", "model", "formula"):
        return "auto"
    raise ValueError(f"identity must be 'leader', 'follower', or 'auto', got {s!r}")


def load_env_from_csv(
    csv_path: str,
    prob_A: float = 0.8,
    prob_B: float = 0.5,
    prob_C: float = 0.2,
) -> pd.DataFrame:
    raw = pd.read_csv(csv_path)
    raw = raw.rename(columns={"x": "Row", "y": "Col", "mine_type": "mine_type", "alien_id": "alien_id"})
    rows = []
    for _, row in raw.iterrows():
        r, c = int(row["Row"]), int(row["Col"])
        mine_type = str(row["mine_type"]).strip() if not pd.isna(row["mine_type"]) else ""
        alien_val = row["alien_id"]

        if mine_type == "Gold Mine A":
            reward_prob, richness = prob_A, "high"
        elif mine_type == "Gold Mine B":
            reward_prob, richness = prob_B, "medium"
        elif mine_type == "Gold Mine C":
            reward_prob, richness = prob_C, "low"
        else:
            mine_type, reward_prob, richness = "", 0.0, "none"

        alien_level = 0 if (pd.isna(alien_val) or str(alien_val).strip() == "") else int(float(alien_val))
        rows.append({
            "Row": r,
            "Col": c,
            "Location": f"{r}:{c}",
            "mine_type": mine_type,
            "richness": richness,
            "reward_prob": float(reward_prob),
            "alien_level": alien_level,
        })
    env = pd.DataFrame(rows)
    env.set_index(["Row", "Col"], inplace=True, drop=False)
    return env


@dataclass
class ForagerConfig:
    moves_per_round: int = 5
    lambda_sec: float = 1.0
    w_scale: float = 1.0
    move_cost: float = 0.0
    mine_capacity_high: int = 8
    mine_capacity_medium: int = 5
    mine_capacity_low: int = 2
    security_pos: Optional[Tuple[int, int]] = None
    seed: Optional[int] = None
    role_mode: str = "auto"
    escort_scan_weight: float = 0.62
    escort_direction_weight: float = 0.42
    escort_scan_radius: int = 2
    escort_follower_only: bool = True
    escort_trust_explore_scale: float = 0.92
    escort_trust_scan_boost: float = 1.38
    escort_trust_min_explore_when_mines: float = 0.3
    follower_ring_distance: int = 3
    follower_sideways_weight: float = 0.14
    follower_explore_scale: float = 0.28
    follower_goal_weight: float = 1.0
    approach_mine_weight: float = 1.15
    known_mine_explore_cap: float = 0.0
    follower_mine_near_security_radius: int = 10
    follower_ring_lambda_scale: float = 0.42
    follower_cleared_trail_weight: float = 0.55
    follower_umbrella_weight: float = 0.24
    follower_mine_progress_weight: float = 2.0
    follower_mine_progress_step_weight: float = 0.75
    follower_mine_progress_max_security_distance: int = 4
    follower_backtrack_penalty: float = 0.45
    follower_target_commit_bonus: float = 0.7
    escort_safe_frontier_weight: float = 0.9
    escort_safe_frontier_neighbor_weight: float = 0.45
    escort_no_mine_explore_boost: float = 0.35
    no_mine_revealed_loop_penalty: float = 1.1
    no_mine_backtrack_extra_penalty: float = 0.55


class ForagerAgent:
    def __init__(self, env: pd.DataFrame, cfg: ForagerConfig):
        self.env = env
        self.cfg = cfg
        self.rng = np.random.default_rng(cfg.seed)
        self.revealed_cells: Set[Tuple[int, int]] = set()

        max_r = int(self.env.index.get_level_values(0).max())
        max_c = int(self.env.index.get_level_values(1).max())
        self._start_pos = (max_r // 2, max_c // 2)
        self.pos = self._start_pos
        self.security_pos = cfg.security_pos or (max_r // 2, max_c // 2)

        self.t = 0
        self.total_reward = 0.0
        self.round_reward = 0.0
        self.visited: Set[Tuple[int, int]] = {self.pos}
        self.stunned_steps = 0

        self.log: List[dict] = []
        self.frames_action: List[str] = []
        self.prev_pos: Optional[Tuple[int, int]] = None
        self.current_round: int = 0

        self.mine_remaining: Dict[Tuple[int, int], int] = {}
        self.initial_mine_remaining: Dict[Tuple[int, int], int] = {}
        for (r, c), row in self.env.iterrows():
            mt = str(row["mine_type"]).strip()
            if mt == "Gold Mine A":
                self.initial_mine_remaining[(r, c)] = int(cfg.mine_capacity_high)
            elif mt == "Gold Mine B":
                self.initial_mine_remaining[(r, c)] = int(cfg.mine_capacity_medium)
            elif mt == "Gold Mine C":
                self.initial_mine_remaining[(r, c)] = int(cfg.mine_capacity_low)
        self.mine_remaining = dict(self.initial_mine_remaining)

        self.security_cleared_alien_cells: Set[Tuple[int, int]] = set()
        self.escort_scan_map: Dict[Tuple[int, int], float] = {}
        self.escort_travel_dr: float = 0.0
        self.escort_travel_dc: float = 0.0
        self._explore_anchor: Optional[Tuple[int, int]] = None
        self.escort_trust_mode: bool = False
        self._committed_mine_target: Optional[Tuple[int, int]] = None
        self._follower_mode: str = "scout"
        self._follower_target: Optional[Tuple[int, int]] = None

    def _ensure_explore_anchor(self) -> None:
        if self._explore_anchor is not None and self._explore_anchor not in self.revealed_cells:
            return
        unrev = [p for p in self.env.index if p not in self.revealed_cells]
        if not unrev:
            self._explore_anchor = None
            return
        if self.escort_trust_mode and self.escort_scan_map:
            hub = max(self.escort_scan_map.items(), key=lambda kv: kv[1])[0]
            self._explore_anchor = min(
                unrev,
                key=lambda p: (
                    abs(p[0] - hub[0]) + abs(p[1] - hub[1]),
                    abs(p[0] - self.pos[0]) + abs(p[1] - self.pos[1]),
                    p[0],
                    p[1],
                ),
            )
            return
        self._explore_anchor = min(
            unrev,
            key=lambda p: (
                abs(p[0] - self.pos[0]) + abs(p[1] - self.pos[1]),
                p[0],
                p[1],
            ),
        )

    def _reveal_here(self) -> None:
        self.revealed_cells.add(tuple(self.pos))

    def current_reward_prob(self) -> float:
        if self.pos in self.mine_remaining and self.mine_remaining[self.pos] <= 0:
            return 0.0
        return _reward_observed(self.env, self.pos, self.revealed_cells)

    def neighbors(self) -> List[Tuple[int, int]]:
        r, c = self.pos
        return [p for p in [(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)] if p in self.env.index]

    def Vdig(self) -> float:
        return self.current_reward_prob()

    def Vmove(self) -> float:
        return self.round_reward / float(self.t) if self.t > 0 else 0.0

    def w_t(self, vd: float, vm: float) -> float:
        delta = vd - vm
        return float(1.0 / (1.0 + np.exp(-max(self.cfg.w_scale, 1e-6) * delta)))

    def _active_dig_mines(self) -> List[Tuple[int, int]]:
        out: List[Tuple[int, int]] = []
        for p in self.revealed_cells:
            if p not in self.env.index or not _mine_observed(self.env, p, self.revealed_cells):
                continue
            if self.mine_remaining.get(p, 0) <= 0:
                continue
            if _reward_observed(self.env, p, self.revealed_cells) <= 0.0:
                continue
            out.append(p)
        return out

    def _exploit_mine_set(self) -> List[Tuple[int, int]]:
        mines = self._active_dig_mines()
        rm = str(self.cfg.role_mode).lower()
        if rm not in ("follow", "follower"):
            return mines
        R = max(0, int(self.cfg.follower_mine_near_security_radius))
        sx, sy = self.security_pos
        local = [p for p in mines if abs(p[0] - sx) + abs(p[1] - sy) <= R]
        return local if local else mines

    def E_exploit(self, s_prime: Tuple[int, int]) -> float:
        best = 0.0
        for p in self._exploit_mine_set():
            rp = _reward_observed(self.env, p, self.revealed_cells)
            d = abs(s_prime[0] - p[0]) + abs(s_prime[1] - p[1])
            best = max(best, rp / (1.0 + float(d)))
        return best

    def _has_active_revealed_mine(self) -> bool:
        return len(self._active_dig_mines()) > 0

    def _tie_break_mine_targets(self) -> List[Tuple[int, int]]:
        mines = self._active_dig_mines()
        rm = str(self.cfg.role_mode).lower()
        if rm not in ("follow", "follower"):
            return mines
        R = max(0, int(self.cfg.follower_mine_near_security_radius))
        sx, sy = self.security_pos
        local = [p for p in mines if abs(p[0] - sx) + abs(p[1] - sy) <= R]
        return local if local else mines

    def E_explore(self, s_prime: Tuple[int, int]) -> float:
        self._ensure_explore_anchor()
        if self._explore_anchor is None:
            return 0.0
        d = abs(s_prime[0] - self._explore_anchor[0]) + abs(s_prime[1] - self._explore_anchor[1])
        return 1.0 / (1.0 + d)

    def _escort_trust_explore_pull(self, s_prime: Tuple[int, int]) -> float:
        if not self.escort_trust_mode or not self.escort_scan_map:
            return 0.0
        s = float(sum(self.escort_scan_map.values()))
        if s <= 0:
            return 0.0
        tot = 0.0
        for (qr, qc), val in self.escort_scan_map.items():
            if val <= 0:
                continue
            d = abs(s_prime[0] - qr) + abs(s_prime[1] - qc)
            tot += float(val) / (1.0 + float(d))
        return float(tot) / s

    def _has_revealed_frontier_neighbor(self, p: Tuple[int, int]) -> bool:
        r, c = p
        for q in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):
            if q in self.env.index and q not in self.revealed_cells:
                return True
        return False

    def _escort_safe_frontier_bonus(self, p: Tuple[int, int]) -> float:
        rm = str(self.cfg.role_mode).lower()
        if rm not in ("follow", "follower"):
            return 0.0
        if self._has_active_revealed_mine():
            return 0.0

        bonus = 0.0
        if p not in self.revealed_cells:
            bonus += float(self.cfg.escort_safe_frontier_weight)
        elif self._has_revealed_frontier_neighbor(p):
            bonus += float(self.cfg.escort_safe_frontier_neighbor_weight)

        if bonus <= 0.0:
            return 0.0

        scan_pull = 0.0
        if self.escort_scan_map:
            total = float(sum(max(0.0, v) for v in self.escort_scan_map.values()))
            if total > 0.0:
                weighted = 0.0
                for q, val in self.escort_scan_map.items():
                    if val <= 0:
                        continue
                    d = abs(p[0] - q[0]) + abs(p[1] - q[1])
                    weighted += float(val) / (1.0 + float(d))
                scan_pull = weighted / total

        cleared_pull = 0.0
        if self.security_cleared_alien_cells:
            best_d = min(
                abs(p[0] - q[0]) + abs(p[1] - q[1])
                for q in self.security_cleared_alien_cells
            )
            cleared_pull = 1.0 / (1.0 + float(best_d))

        sec_d = abs(p[0] - self.security_pos[0]) + abs(p[1] - self.security_pos[1])
        near_security = 1.0 / (1.0 + float(sec_d))

        safety = max(scan_pull, cleared_pull, near_security)
        return bonus * (1.0 + safety)

    def _no_mine_loop_penalty(self, p: Tuple[int, int]) -> float:
        rm = str(self.cfg.role_mode).lower()
        if rm not in ("follow", "follower"):
            return 0.0
        if self._has_active_revealed_mine():
            return 0.0

        penalty = 0.0
        if p in self.revealed_cells and not self._has_revealed_frontier_neighbor(p):
            penalty += float(self.cfg.no_mine_revealed_loop_penalty)
        if self.prev_pos is not None and p == self.prev_pos:
            penalty += float(self.cfg.no_mine_backtrack_extra_penalty)
        return penalty

    def A_goal(self, s_prime: Tuple[int, int], w: float) -> float:
        ee = self.E_explore(s_prime)
        rm = str(self.cfg.role_mode).lower()
        if rm in ("follow", "follower"):
            ee *= float(self.cfg.follower_explore_scale)
            if self.escort_trust_mode:
                pull = self._escort_trust_explore_pull(s_prime)
                ee = max(ee, pull * float(self.cfg.escort_trust_explore_scale))
                if not self._has_active_revealed_mine():
                    ee += float(self.cfg.escort_no_mine_explore_boost) * self._escort_safe_frontier_bonus(s_prime)
        w_use = float(w)
        if self._has_active_revealed_mine():
            cap = float(self.cfg.known_mine_explore_cap)
            if self.escort_trust_mode and rm in ("follow", "follower"):
                cap = max(cap, float(self.cfg.escort_trust_min_explore_when_mines))
            w_use = min(w_use, cap)
        return (1.0 - w_use) * self.E_exploit(s_prime) + w_use * ee

    def _A_goal_as_leader(self, s_prime: Tuple[int, int], w: float) -> float:
        ee = self.E_explore(s_prime)
        w_use = float(w)
        if self._has_active_revealed_mine():
            w_use = min(w_use, float(self.cfg.known_mine_explore_cap))
        return (1.0 - w_use) * self.E_exploit(s_prime) + w_use * ee

    def leader_style_neighbor_value(
        self,
        p: Tuple[int, int],
        w: float,
        spread_from: Tuple[int, int],
        move_cost: float,
    ) -> float:
        lam = abs(float(self.cfg.lambda_sec)) if float(self.cfg.lambda_sec) != 0 else 1.0
        a = self._A_goal_as_leader(p, w)
        d = abs(p[0] - spread_from[0]) + abs(p[1] - spread_from[1])
        mine_pull = float(self.cfg.approach_mine_weight) * self.E_exploit(p)
        return lam * float(d) * a + mine_pull - float(move_cost)

    def _escort_neighbor_bonus(self, p: Tuple[int, int]) -> float:
        if self.cfg.escort_scan_weight <= 0.0 and self.cfg.escort_direction_weight <= 0.0:
            return 0.0
        rm = str(self.cfg.role_mode).lower()
        if self.cfg.escort_follower_only and rm not in ("follow", "follower"):
            return 0.0
        scan_part = 0.0
        rad = max(0, int(self.cfg.escort_scan_radius))
        for (qr, qc), val in self.escort_scan_map.items():
            dqp = abs(qr - p[0]) + abs(qc - p[1])
            if dqp <= rad:
                scan_part += float(val) / (1.0 + float(dqp))
        tr, tc = self.escort_travel_dr, self.escort_travel_dc
        norm = float(np.hypot(tr, tc))
        dir_part = 0.0
        if norm > 1e-6:
            ur, uc = tr / norm, tc / norm
            vr, vc = float(p[0] - self.pos[0]), float(p[1] - self.pos[1])
            dir_part = max(0.0, vr * ur + vc * uc)
        out = (
            float(self.cfg.escort_scan_weight) * scan_part
            + float(self.cfg.escort_direction_weight) * dir_part
        )
        if getattr(self, "escort_trust_mode", False):
            out *= float(self.cfg.escort_trust_scan_boost)
        return out

    def _follower_sideways_bonus(self, p: Tuple[int, int]) -> float:
        rm = str(self.cfg.role_mode).lower()
        if rm not in ("follow", "follower"):
            return 0.0
        tr, tc = self.security_pos
        sr, sc = self.pos
        pr, pc = p
        rx, ry = sr - tr, sc - tc
        sx, sy = pr - sr, pc - sc
        cross = float(rx * sy - ry * sx)
        return float(self.cfg.follower_sideways_weight) * (abs(cross) / 2.0)

    def _follower_corridor_bonus(self, p: Tuple[int, int]) -> float:
        rm = str(self.cfg.role_mode).lower()
        if rm not in ("follow", "follower"):
            return 0.0
        b = 0.0
        if p in self.security_cleared_alien_cells:
            b += float(self.cfg.follower_cleared_trail_weight)
        ds = abs(p[0] - self.security_pos[0]) + abs(p[1] - self.security_pos[1])
        b += float(self.cfg.follower_umbrella_weight) / (1.0 + float(ds))
        return b

    def _best_revealed_mine(self) -> Optional[Tuple[int, int]]:
        mines = self._exploit_mine_set()
        if not mines:
            self._committed_mine_target = None
            return None
        if self._committed_mine_target in mines:
            committed_reward = _reward_observed(self.env, self._committed_mine_target, self.revealed_cells)
        else:
            committed_reward = -1.0
        sx, sy = self.security_pos
        best = max(
            mines,
            key=lambda q: (
                _reward_observed(self.env, q, self.revealed_cells),
                -(abs(q[0] - self.pos[0]) + abs(q[1] - self.pos[1])),
                -(abs(q[0] - sx) + abs(q[1] - sy)),
                -q[0],
                -q[1],
            ),
        )
        best_reward = _reward_observed(self.env, best, self.revealed_cells)
        if (
            self._committed_mine_target in mines
            and committed_reward >= best_reward - 0.15
        ):
            return self._committed_mine_target
        self._committed_mine_target = best
        return best

    def _safe_mine_target_for_follower(self) -> Optional[Tuple[int, int]]:
        # Follower harvests only mines that remain under the escort's protective umbrella.
        # This keeps the role "human-like": if security revealed something good nearby, go take it;
        # if the mine is too far from the escort, do not break formation just because the reward is high.
        target = self._best_revealed_mine()
        if target is None:
            return None
        sec_d = abs(target[0] - self.security_pos[0]) + abs(target[1] - self.security_pos[1])
        max_sec_d = max(
            int(self.cfg.follower_mine_progress_max_security_distance),
            int(self.cfg.follower_ring_distance) + 1,
        )
        if sec_d > max_sec_d:
            return None
        return target

    def _frontier_candidates(self) -> List[Tuple[int, int]]:
        # Frontier means "still useful for discovery": either the cell itself is fogged, or it borders
        # fog. Cells that are fully revealed and surrounded by revealed neighbors are poor scouting
        # targets and are excluded here so the follower does not loop in already-known safe space.
        out: List[Tuple[int, int]] = []
        for p in self.env.index:
            if p in self.revealed_cells and not self._has_revealed_frontier_neighbor(p):
                continue
            out.append(p)
        return out

    def _best_safe_frontier_target(self) -> Optional[Tuple[int, int]]:
        # When security has not surfaced a mine, the follower helps by scouting near the escort rather
        # than passively orbiting. We choose one frontier target near the scan trail / cleared corridor
        # and keep it for a few steps, which produces more intentional motion than re-scoring neighbors
        # every step.
        frontier = self._frontier_candidates()
        if not frontier:
            return None
        best = None
        best_score = -np.inf
        for p in frontier:
            sec_d = abs(p[0] - self.security_pos[0]) + abs(p[1] - self.security_pos[1])
            if sec_d > max(int(self.cfg.follower_ring_distance) + 2, int(self.cfg.follower_mine_progress_max_security_distance) + 1):
                continue
            scan_pull = self._escort_trust_explore_pull(p) if self.escort_scan_map else 0.0
            cleared_pull = 0.0
            if self.security_cleared_alien_cells:
                best_clear_d = min(abs(p[0] - q[0]) + abs(p[1] - q[1]) for q in self.security_cleared_alien_cells)
                cleared_pull = 1.0 / (1.0 + float(best_clear_d))
            frontier_bonus = 1.0 if p not in self.revealed_cells else 0.5
            dist_from_me = abs(p[0] - self.pos[0]) + abs(p[1] - self.pos[1])
            score = 1.4 * frontier_bonus + 1.1 * max(scan_pull, cleared_pull) + 0.5 / (1.0 + float(dist_from_me)) + 0.35 / (1.0 + float(sec_d))
            if score > best_score:
                best_score = score
                best = p
        return best

    def _choose_follower_intent(self) -> Tuple[str, Optional[Tuple[int, int]]]:
        # High-level follower state machine:
        # 1. harvest: escort has revealed a nearby safe mine -> commit to collecting it
        # 2. rejoin: follower drifted too far from security -> close distance first
        # 3. scout: no mine to harvest and distance is acceptable -> reveal fog near escort path
        #
        # This is the main behavioral shift away from dense reward shaping. Instead of asking
        # "which adjacent square scores highest right now?", the follower first decides what it is
        # trying to accomplish over the next few steps, then walks toward that target consistently.
        dist_to_security = abs(self.pos[0] - self.security_pos[0]) + abs(self.pos[1] - self.security_pos[1])
        max_rejoin = max(int(self.cfg.follower_ring_distance) + 2, int(self.cfg.follower_mine_progress_max_security_distance) + 1)
        mine_target = self._safe_mine_target_for_follower()
        if mine_target is not None:
            return "harvest", mine_target
        if dist_to_security > max_rejoin:
            return "rejoin", self.security_pos
        frontier_target = self._best_safe_frontier_target()
        if frontier_target is not None:
            return "scout", frontier_target
        return "rejoin", self.security_pos

    def _follower_target_still_valid(self) -> bool:
        # Each mode keeps its target until it stops making sense:
        # - harvest: the chosen mine is still the right safe mine to collect
        # - scout: the target is still frontier-like (unrevealed or bordering fog)
        # - rejoin: stay in rejoin until we are back near the escort band
        #
        # This target persistence is what removes the "circle and wobble" look from the follower.
        if self._follower_target is None:
            return False
        if self._follower_mode == "harvest":
            return self._follower_target == self._safe_mine_target_for_follower()
        if self._follower_mode == "scout":
            return (
                self._follower_target in self.env.index
                and (
                    self._follower_target not in self.revealed_cells
                    or self._has_revealed_frontier_neighbor(self._follower_target)
                )
            )
        if self._follower_mode == "rejoin":
            dist_to_security = abs(self.pos[0] - self.security_pos[0]) + abs(self.pos[1] - self.security_pos[1])
            return dist_to_security > int(self.cfg.follower_ring_distance)
        return False

    def _step_toward_target(self, target: Tuple[int, int]) -> Tuple[bool, int]:
        # Once the follower has intent, movement becomes simple and purposeful:
        # - choose legal neighbors
        # - prefer ones that reduce distance to the committed target
        # - break ties differently for each mode:
        #   scout   -> prefer fog/frontier near escort-cleared areas
        #   harvest -> prefer stronger progress toward the chosen mine
        #   rejoin  -> prefer landing back on the escort ring
        #
        # This produces more human-looking motion than a single blended score for all situations.
        nbrs = self.neighbors()
        if not nbrs:
            return False, 0
        legal = [p for p in nbrs if self._can_step_onto_security_cell(p)]
        if not legal:
            return False, 0
        if len(legal) > 1 and self.prev_pos is not None:
            alt = [p for p in legal if p != self.prev_pos]
            if alt:
                legal = alt
        best_d = min(abs(p[0] - target[0]) + abs(p[1] - target[1]) for p in legal)
        candidates = [p for p in legal if abs(p[0] - target[0]) + abs(p[1] - target[1]) == best_d]
        if self._follower_mode == "scout":
            best_frontier = max(
                candidates,
                key=lambda p: (
                    p not in self.revealed_cells,
                    self._has_revealed_frontier_neighbor(p),
                    self._escort_safe_frontier_bonus(p),
                    -abs(p[0] - self.security_pos[0]) - abs(p[1] - self.security_pos[1]),
                    -p[0],
                    -p[1],
                ),
            )
            best_p = best_frontier
        elif self._follower_mode == "harvest":
            best_p = max(
                candidates,
                key=lambda p: (
                    self._follower_mine_progress_bonus(p),
                    self.E_exploit(p),
                    -abs(p[0] - self.security_pos[0]) - abs(p[1] - self.security_pos[1]),
                    -p[0],
                    -p[1],
                ),
            )
        else:
            ring = int(self.cfg.follower_ring_distance)
            best_p = min(
                candidates,
                key=lambda p: (
                    abs((abs(p[0] - self.security_pos[0]) + abs(p[1] - self.security_pos[1])) - ring),
                    abs(p[0] - self.security_pos[0]) + abs(p[1] - self.security_pos[1]),
                    p[0],
                    p[1],
                ),
            )
        self.pos = best_p
        self.visited.add(self.pos)
        self._reveal_here()
        al = int(self.env.loc[self.pos, "alien_level"])
        if (
            al > 0
            and self.pos != self.security_pos
            and self.pos not in self.security_cleared_alien_cells
        ):
            self.stunned_steps = 3
        return True, al

    def _move_as_follower_behavior(self, w: float) -> Tuple[bool, int]:
        # ``w`` still exists for compatibility with the rest of the forager interface, but the
        # follower policy no longer uses the explore/exploit logistic directly. The follower now
        # acts more like a teammate with short-term intent:
        # - commit to harvest if security just revealed a safe mine
        # - otherwise scout near the escort's path to reveal fog
        # - rejoin if it drifts too far from the protector
        del w
        if not self._follower_target_still_valid():
            mode, target = self._choose_follower_intent()
            self._follower_mode = mode
            self._follower_target = target
        if self._follower_target is None:
            return False, 0
        moved, al = self._step_toward_target(self._follower_target)
        if self._follower_mode == "harvest" and self.pos == self._follower_target:
            self._committed_mine_target = self._follower_target
        elif self._follower_mode == "scout" and (
            self.pos == self._follower_target or self._follower_target in self.revealed_cells and not self._has_revealed_frontier_neighbor(self._follower_target)
        ):
            self._follower_target = None
        elif self._follower_mode == "rejoin":
            dist_to_security = abs(self.pos[0] - self.security_pos[0]) + abs(self.pos[1] - self.security_pos[1])
            if dist_to_security <= int(self.cfg.follower_ring_distance):
                self._follower_target = None
        return moved, al

    def _follower_mine_progress_bonus(self, p: Tuple[int, int]) -> float:
        rm = str(self.cfg.role_mode).lower()
        if rm not in ("follow", "follower"):
            return 0.0
        target = self._best_revealed_mine()
        if target is None:
            return 0.0

        cur_d = abs(self.pos[0] - target[0]) + abs(self.pos[1] - target[1])
        nxt_d = abs(p[0] - target[0]) + abs(p[1] - target[1])
        if nxt_d >= cur_d:
            return 0.0

        sec_d = abs(p[0] - self.security_pos[0]) + abs(p[1] - self.security_pos[1])
        max_sec_d = max(
            int(self.cfg.follower_mine_progress_max_security_distance),
            int(self.cfg.follower_ring_distance) + 1,
        )
        if sec_d > max_sec_d:
            return 0.0

        reward = _reward_observed(self.env, target, self.revealed_cells)
        progress = float(cur_d - nxt_d)
        bonus = (
            float(self.cfg.follower_mine_progress_weight) * reward
            + float(self.cfg.follower_mine_progress_step_weight) * progress
        )
        if target == self._committed_mine_target:
            bonus += float(self.cfg.follower_target_commit_bonus)
        return bonus

    def _can_step_onto_security_cell(self, p: Tuple[int, int]) -> bool:
        if p != self.security_pos:
            return True
        if p not in self.revealed_cells:
            return False
        if not _mine_observed(self.env, p, self.revealed_cells):
            return False
        return self.mine_remaining.get(p, 0) > 0

    def _lambda_for_moves(self) -> float:
        lam = float(self.cfg.lambda_sec)
        rm = str(self.cfg.role_mode).lower()
        if rm in ("follow", "follower"):
            return -abs(lam) if lam != 0 else -1.0
        if rm in ("lead", "leader"):
            return abs(lam) if lam != 0 else 1.0
        return lam

    def _w_identity_adjust(self, w: float) -> float:
        rm = str(self.cfg.role_mode).lower()
        if rm in ("lead", "leader"):
            return float(min(1.0, w + 0.2))
        if rm in ("follow", "follower"):
            w = float(max(0.0, w - 0.2))
            if getattr(self, "escort_trust_mode", False):
                w = float(min(1.0, w + 0.1))
            return w
        return w

    def _log(self, **kw):
        kw.setdefault("step", self.t)
        kw.setdefault("row", self.pos[0])
        kw.setdefault("col", self.pos[1])
        kw.setdefault("total_reward", self.total_reward)
        kw.setdefault("round_reward", self.round_reward)
        kw.setdefault("round", self.current_round)
        kw.setdefault("forager_role_mode", self.cfg.role_mode)
        self.log.append(kw)

    def reset_segment_reward(self):
        self.round_reward = 0.0

    def _dig(self) -> float:
        r = self.current_reward_prob()
        self.total_reward += r
        self.round_reward += r
        if self.pos in self.mine_remaining and self.mine_remaining[self.pos] > 0:
            self.mine_remaining[self.pos] -= 1
        return r

    def _move_to_best_A(self, w: float) -> Tuple[bool, int]:
        nbrs = self.neighbors()
        if not nbrs:
            return False, 0
        lam = self._lambda_for_moves()
        rm = str(self.cfg.role_mode).lower()
        follower = rm in ("follow", "follower")
        if follower:
            return self._move_as_follower_behavior(w)
        r_ring = max(1, int(self.cfg.follower_ring_distance))
        scored: List[Tuple[float, Tuple[int, int]]] = []
        for p in nbrs:
            if not self._can_step_onto_security_cell(p):
                continue
            a = self.A_goal(p, w)
            d = abs(p[0] - self.security_pos[0]) + abs(p[1] - self.security_pos[1])
            esc = self._escort_neighbor_bonus(p)
            if follower:
                ring_err = abs(float(d - r_ring))
                ring_bonus = 0.12 * float(d == r_ring)
                side = self._follower_sideways_bonus(p)
                rscale = float(self.cfg.follower_ring_lambda_scale)
                mine_progress = self._follower_mine_progress_bonus(p)
                backtrack_penalty = 0.0
                if self.prev_pos is not None and p == self.prev_pos:
                    backtrack_penalty = float(self.cfg.follower_backtrack_penalty)
                v = (
                    lam * ring_err * rscale
                    + float(self.cfg.follower_goal_weight) * a
                    + esc
                    + side
                    + ring_bonus
                    + self._follower_corridor_bonus(p)
                    + self._escort_safe_frontier_bonus(p)
                    + mine_progress
                    - backtrack_penalty
                    - self._no_mine_loop_penalty(p)
                    - self.cfg.move_cost
                )
            else:
                mine_pull = float(self.cfg.approach_mine_weight) * self.E_exploit(p)
                v = lam * float(d) * a + mine_pull + esc - self.cfg.move_cost
            scored.append((v, p))
        if follower and getattr(self, "escort_trust_mode", False) and self.prev_pos is not None:
            alt = [(v, p) for v, p in scored if p != self.prev_pos]
            if alt:
                scored = alt
        if not scored:
            return False, 0
        best_val = max(s[0] for s in scored)
        candidates = [p for v, p in scored if v >= best_val - 1e-12]
        if len(candidates) > 1 and self.prev_pos is not None and self.prev_pos in candidates:
            alt = [p for p in candidates if p != self.prev_pos]
            if alt:
                candidates = alt
        if len(candidates) > 1 and self._has_active_revealed_mine():
            targets = self._tie_break_mine_targets()
            if targets:
                def _dist_mine(q: Tuple[int, int]) -> int:
                    return min(abs(q[0] - t[0]) + abs(q[1] - t[1]) for t in targets)

                best_d = min(_dist_mine(p) for p in candidates)
                candidates = [p for p in candidates if _dist_mine(p) == best_d]
        best_p = min(candidates, key=lambda p: (p[0], p[1]))
        if follower:
            self._committed_mine_target = self._best_revealed_mine()
        self.pos = best_p
        self.visited.add(self.pos)
        self._reveal_here()
        al = int(self.env.loc[self.pos, "alien_level"])
        if (
            al > 0
            and self.pos != self.security_pos
            and self.pos not in self.security_cleared_alien_cells
        ):
            self.stunned_steps = 3
        return True, al

    def step(self) -> str:
        pos_before = tuple(self.pos)
        self._reveal_here()
        if self.stunned_steps > 0:
            self._log(action="stunned", decision="stunned", Vdig=np.nan, Vmove=np.nan, w=np.nan)
            self.frames_action.append("stunned")
            self.stunned_steps -= 1
            self.t += 1
            self.prev_pos = pos_before
            return "stunned"

        if self.current_reward_prob() > 0.0:
            vd, vm = self.Vdig(), self.Vmove()
            w = self._w_identity_adjust(self.w_t(vd, vm))
            r_gained = self._dig()
            self._log(action="dig", decision="dig", Vdig=vd, Vmove=vm, w=w, r_gained=r_gained)
            self.frames_action.append("dig")
            self.t += 1
            self.prev_pos = pos_before
            return "dig"

        vd, vm = self.Vdig(), self.Vmove()
        w = self._w_identity_adjust(self.w_t(vd, vm))
        force_move = (
            self.current_reward_prob() == 0.0
            and (
                any(_reward_observed(self.env, p, self.revealed_cells) > 0 for p in self.neighbors())
                or any(p not in self.revealed_cells for p in self.env.index)
            )
        )
        if (vd > vm) and not force_move:
            self._dig()
            self._log(action="dig", decision="dig", Vdig=vd, Vmove=vm, w=w)
            self.frames_action.append("dig")
        else:
            moved, alien_lvl = self._move_to_best_A(w)
            self._log(
                action="move" if moved else "no_move",
                decision="move",
                Vdig=vd,
                Vmove=vm,
                w=w,
                alien_level=alien_lvl if alien_lvl else None,
            )
            self.frames_action.append("move" if moved else "no_move")
        self.t += 1
        self.prev_pos = pos_before
        return self.frames_action[-1]


@dataclass
class SecurityConfig:
    vtype_threshold: float = 0.5
    gamma: float = 0.15
    lead_weight_init: float = 0.5
    follow_weight_init: float = 0.5
    window_size: int = 10
    e_scan: float = 1.0
    follow_radius: int = 3
    strategy_if_tie: str = "follow"
    seed: Optional[int] = None
    chase_belief_threshold: float = 0.22
    chase_clear_radius: int = 3
    role_mode: str = "auto"
    lead_reward_bias: float = 0.26
    lead_mine_attraction: float = 0.28
    lead_reward_pool_radius: int = 3
    leader_backtrack_penalty: float = 0.42
    leader_frontier_weight: float = 0.2
    leader_threat_weight: float = 0.16
    scan_spread_radius: int = 2
    lead_target_stickiness: float = 0.12
    recent_cell_penalty: float = 0.14


class SecurityAgent:
    def __init__(self, env, cfg: SecurityConfig, start_pos: Tuple[int, int]):
        self.env = env
        self.cfg = cfg
        self.rng = np.random.default_rng(cfg.seed)
        self._start_pos = start_pos
        self.pos = start_pos
        self.revealed_cells: Set[Tuple[int, int]] = set()

        max_r = int(self.env.index.get_level_values(0).max())
        max_c = int(self.env.index.get_level_values(1).max())
        self.Dmax = max(1, max_r + max_c)

        self.lead_weight = float(cfg.lead_weight_init)
        self.follow_weight = float(cfg.follow_weight_init)
        self.vmove_lead_hist: deque = deque(maxlen=cfg.window_size)
        self.vmove_follow_hist: deque = deque(maxlen=cfg.window_size)

        self.log: List[dict] = []
        self._prev_security_pos: Optional[Tuple[int, int]] = None
        self._next_beat_is_role_move: bool = True
        self.alien_cells_cleared: Set[Tuple[int, int]] = set()
        self.scan_accum: Dict[Tuple[int, int], float] = {}
        self.travel_dr: float = 0.0
        self.travel_dc: float = 0.0
        self._travel_ema_eta: float = 0.45
        self._gold_mine_cell_count: int = sum(
            1 for (_, row) in self.env.iterrows() if str(row["mine_type"]).strip() != ""
        )
        self._committed_lead_target: Optional[Tuple[int, int]] = None
        self._recent_positions: deque = deque([start_pos], maxlen=4)

    def _count_revealed_gold_mines(self) -> int:
        n = 0
        for p in self.revealed_cells:
            if p not in self.env.index:
                continue
            if str(self.env.loc[p, "mine_type"]).strip() != "":
                n += 1
        return n

    def _neighbors(self, pos: Tuple[int, int]) -> List[Tuple[int, int]]:
        r, c = pos
        return [p for p in [(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)] if p in self.env.index]

    def _manhattan(self, a: Tuple[int, int], b: Tuple[int, int]) -> int:
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    def _forager_velocity(
        self,
        forager_pos: Tuple[int, int],
        forager_pos_before: Optional[Tuple[int, int]],
    ) -> Tuple[int, int]:
        if forager_pos_before is None:
            return (0, 0)
        return (forager_pos[0] - forager_pos_before[0], forager_pos[1] - forager_pos_before[1])

    def _step_toward(
        self,
        target: Tuple[int, int],
        exclude: Optional[Tuple[int, int]] = None,
        forbid_cell: Optional[Tuple[int, int]] = None,
        velocity_hint: Optional[Tuple[int, int]] = None,
    ) -> bool:
        # Escort "greedy toward target" step with anti-wobble tie-breaking.
        # The key detail is that recently occupied cells lose tie-breaks when another step is just as
        # good, so the security agent does not bounce between two cells unless there is a real reason.
        if self.pos == target:
            return False
        nbrs = self._neighbors(self.pos)
        if forbid_cell is not None:
            nbrs = [p for p in nbrs if p != forbid_cell]
        if not nbrs:
            return False
        best_d = min(self._manhattan(p, target) for p in nbrs)
        best = [p for p in nbrs if self._manhattan(p, target) == best_d]
        if exclude is not None and len(best) > 1:
            alt = [p for p in best if p != exclude]
            if alt:
                best = alt
        if velocity_hint is not None:
            vr, vc = velocity_hint
            if abs(vr) + abs(vc) > 0:
                sr, sc = self.pos
                dots = [(p, (p[0] - sr) * vr + (p[1] - sc) * vc) for p in best]
                mx = max(d for _, d in dots)
                best = [p for p, d in dots if d == mx]
        if len(best) > 1:
            best = sorted(
                best,
                key=lambda p: (
                    sum(1 for q in self._recent_positions if q == p),
                    p[0],
                    p[1],
                ),
            )
        self.pos = min(best, key=lambda p: (p[0], p[1]))
        return True

    def _orbit_cross_turn(self, target: Tuple[int, int], p: Tuple[int, int]) -> int:
        tr, tc = target
        sr, sc = self.pos
        pr, pc = p
        rx, ry = sr - tr, sc - tc
        sx, sy = pr - sr, pc - sc
        return rx * sy - ry * sx

    def _orbit_near(
        self,
        target: Tuple[int, int],
        radius: int,
        exclude: Optional[Tuple[int, int]] = None,
    ) -> bool:
        # Follow mode is still an orbit, but not a random one: inside the escort radius, prefer moves
        # that preserve a patrol band around the forager while also avoiding very recent cells. That
        # keeps the escort nearby without creating a visible two-cell ping-pong.
        raw = self._neighbors(self.pos)
        if not raw:
            return False
        raw = [p for p in raw if p != target]
        if not raw:
            return False
        if self._manhattan(self.pos, target) > radius:
            return self._step_toward(target, exclude=exclude, forbid_cell=target, velocity_hint=None)

        cands = [p for p in raw if exclude is None or p != exclude]
        if not cands:
            cands = list(raw)
        in_ring = [p for p in cands if self._manhattan(p, target) <= radius]
        pool = in_ring if in_ring else cands
        sd = self._manhattan(self.pos, target)

        def tier(p: Tuple[int, int]) -> int:
            d = self._manhattan(p, target)
            if sd == 1:
                if d == 2:
                    return 0
                if d >= 3:
                    return 1
                return 2
            return abs(d - 1)

        def sort_key(p: Tuple[int, int]) -> Tuple[int, int, int, int]:
            t = tier(p)
            cr = self._orbit_cross_turn(target, p)
            recent = sum(1 for q in self._recent_positions if q == p)
            return (t, recent, -cr, p[0], p[1])

        self.pos = min(pool, key=sort_key)
        return True

    def _alien_belief_at(self, pos: Tuple[int, int]) -> float:
        base_reward = _reward_observed(self.env, pos, self.revealed_cells)
        return float(np.clip(0.35 + 0.65 * base_reward, 0.0, 1.0))

    def _register_chase_clearance(self) -> None:
        r0 = int(self.cfg.chase_clear_radius)
        for (r, c), row in self.env.iterrows():
            if int(row["alien_level"]) <= 0:
                continue
            if self._manhattan(self.pos, (r, c)) <= r0:
                self.alien_cells_cleared.add((r, c))

    def _nearest_richest_mine_cell(self, forager_pos: Tuple[int, int]) -> Optional[Tuple[int, int]]:
        best_rp = -1.0
        ties: List[Tuple[int, int]] = []
        for (r, c) in self.revealed_cells:
            if (r, c) not in self.env.index:
                continue
            row = self.env.loc[(r, c)]
            if str(row["mine_type"]).strip() == "":
                continue
            rp = _reward_observed(self.env, (r, c), self.revealed_cells)
            if rp > best_rp + 1e-12:
                best_rp = rp
                ties = [(r, c)]
            elif abs(rp - best_rp) <= 1e-9:
                ties.append((r, c))
        if not ties:
            return None
        return min(ties, key=lambda x: self._manhattan(x, forager_pos))

    def compute_vtype(
        self,
        forager_pos: Tuple[int, int],
        revealed_count: int,
        total_cells: int,
        stunned_count_total: int,
        global_step: int,
    ) -> Dict[str, float]:
        distance_t = self._manhattan(forager_pos, self.pos)
        distance_norm_t = float(np.clip(distance_t / self.Dmax, 0.0, 1.0))
        area_revealed_t = float(np.clip(revealed_count / max(1, total_cells), 0.0, 1.0))
        stun_rate_t = float(np.clip(stunned_count_total / max(1, global_step + 1), 0.0, 1.0))
        vtype_t = distance_norm_t * area_revealed_t * (1.0 - stun_rate_t)
        vtype_t = float(np.clip(vtype_t, 0.0, 1.0))
        return {
            "distance_t": distance_t,
            "distance_norm_t": distance_norm_t,
            "area_revealed_t": area_revealed_t,
            "stun_rate_t": stun_rate_t,
            "vtype_t": vtype_t,
        }

    def _lead_target_cell(
        self,
        forager_pos: Tuple[int, int],
        forager_pos_before: Optional[Tuple[int, int]],
    ) -> Tuple[int, int]:
        # Lead mode uses a sticky target rather than re-deriving a completely fresh destination every
        # step. We still consider extrapolated forager motion, observed reward, and nearest rich mine,
        # but if the previously chosen lead target remains nearly as good, we keep it. That makes the
        # escort look like it is following a plan instead of twitching between adjacent alternatives.
        cur = forager_pos
        if forager_pos_before is not None:
            dr = cur[0] - forager_pos_before[0]
            dc = cur[1] - forager_pos_before[1]
            predicted = (cur[0] + dr, cur[1] + dc)
            if predicted not in self.env.index:
                predicted = cur
        else:
            predicted = cur
        mine_goal = self._nearest_richest_mine_cell(forager_pos)
        alpha = float(np.clip(self.cfg.lead_reward_bias, 0.0, 1.0))
        beta = float(np.clip(self.cfg.lead_mine_attraction, 0.0, 1.0))
        gamma = max(0.0, 1.0 - alpha - beta)
        R = max(0, int(self.cfg.lead_reward_pool_radius))
        Rm = max(R, 3)
        pr, pc = predicted
        candidates: Set[Tuple[int, int]] = {predicted}
        for (r, c) in self.env.index:
            if abs(r - pr) + abs(c - pc) <= R:
                candidates.add((r, c))
        if mine_goal is not None:
            candidates.add(mine_goal)
            gr, gc = mine_goal
            for (r, c) in self.env.index:
                if abs(r - gr) + abs(c - gc) <= Rm:
                    candidates.add((r, c))
        best = predicted
        best_score = -np.inf
        committed_score = -np.inf
        for (r, c) in candidates:
            rp = _reward_observed(self.env, (r, c), self.revealed_cells)
            dp = abs(r - pr) + abs(c - pc)
            align_p = 1.0 / (1.0 + float(dp))
            if mine_goal is not None:
                dm = abs(r - mine_goal[0]) + abs(c - mine_goal[1])
                align_m = 1.0 / (1.0 + float(dm))
            else:
                align_m = 0.0
            score = gamma * align_p + alpha * rp + beta * align_m
            if score > best_score:
                best_score = score
                best = (r, c)
            if self._committed_lead_target == (r, c):
                committed_score = score
        if (
            self._committed_lead_target is not None
            and self._committed_lead_target in candidates
            and committed_score >= best_score - float(self.cfg.lead_target_stickiness)
        ):
            best = self._committed_lead_target
        self._committed_lead_target = best
        return best

    def _step_like_forager_leader(
        self,
        forager_agent: ForagerAgent,
        forager_pos: Tuple[int, int],
        exclude: Optional[Tuple[int, int]],
    ) -> Tuple[bool, str]:
        vd, vm = forager_agent.Vdig(), forager_agent.Vmove()
        w = float(min(1.0, forager_agent.w_t(vd, vm) + 0.2))
        mc = float(forager_agent.cfg.move_cost)
        nbrs = self._neighbors(self.pos)
        scored: List[Tuple[float, Tuple[int, int]]] = []
        bt = float(self.cfg.leader_backtrack_penalty)
        fw = float(self.cfg.leader_frontier_weight)
        tw = float(self.cfg.leader_threat_weight)
        for p in nbrs:
            if p == forager_pos:
                continue
            v = forager_agent.leader_style_neighbor_value(p, w, forager_pos, mc)
            if exclude is not None and p == exclude:
                v -= bt
            if p not in self.revealed_cells:
                v += fw
            v += tw * self._alien_belief_at(p)
            v -= float(self.cfg.recent_cell_penalty) * sum(1 for q in self._recent_positions if q == p)
            scored.append((v, p))
        if not scored:
            return False, "lead_no_step"
        best_val = max(s[0] for s in scored)
        candidates = [p for v, p in scored if v >= best_val - 1e-12]
        if len(candidates) > 1 and exclude is not None and exclude in candidates:
            alt = [p for p in candidates if p != exclude]
            if alt:
                candidates = alt
        if len(candidates) > 1 and forager_agent._has_active_revealed_mine():
            targets = forager_agent._tie_break_mine_targets()
            if targets:
                def _dist_mine(q: Tuple[int, int]) -> int:
                    return min(abs(q[0] - t[0]) + abs(q[1] - t[1]) for t in targets)

                best_d = min(_dist_mine(p) for p in candidates)
                candidates = [p for p in candidates if _dist_mine(p) == best_d]
        best_p = min(candidates, key=lambda p: (p[0], p[1]))
        self.pos = best_p
        return True, "move_lead_forager_style"

    def _update_escort_trace(self, pos_before_step: Tuple[int, int], vscan_t: float) -> None:
        cur = tuple(self.pos)
        dr = int(cur[0]) - int(pos_before_step[0])
        dc = int(cur[1]) - int(pos_before_step[1])
        if abs(dr) + abs(dc) > 0:
            eta = float(self._travel_ema_eta)
            self.travel_dr = (1.0 - eta) * self.travel_dr + eta * float(dr)
            self.travel_dc = (1.0 - eta) * self.travel_dc + eta * float(dc)
        spread = max(0, int(self.cfg.scan_spread_radius))
        vs = float(vscan_t)
        if vs <= 0.0 and spread == 0:
            return
        cr, cc = int(cur[0]), int(cur[1])
        for r_off in range(-spread, spread + 1):
            for c_off in range(-spread, spread + 1):
                if abs(r_off) + abs(c_off) > spread:
                    continue
                r, c = cr + r_off, cc + c_off
                if (r, c) not in self.env.index:
                    continue
                dist = abs(r_off) + abs(c_off)
                w = vs / (1.0 + float(dist))
                self.scan_accum[(r, c)] = self.scan_accum.get((r, c), 0.0) + w

    def _lead_parallel_in_band(
        self,
        forager_pos: Tuple[int, int],
        forager_pos_before: Optional[Tuple[int, int]],
        exclude: Optional[Tuple[int, int]],
    ) -> bool:
        vr, vc = self._forager_velocity(forager_pos, forager_pos_before)
        if abs(vr) + abs(vc) == 0:
            return False
        cand = (self.pos[0] + vr, self.pos[1] + vc)
        if cand not in self.env.index or cand == forager_pos:
            return False
        nbrs = self._neighbors(self.pos)
        if cand not in nbrs:
            return False
        if exclude is not None and cand == exclude:
            return False
        self.pos = cand
        return True

    def _role_move_one_step(
        self,
        forager_agent: ForagerAgent,
        forager_pos: Tuple[int, int],
        forager_pos_before: Optional[Tuple[int, int]],
        exclude: Optional[Tuple[int, int]],
        final_role: str,
    ) -> Tuple[bool, str]:
        vel = self._forager_velocity(forager_pos, forager_pos_before)
        sec_rm = str(self.cfg.role_mode).lower()
        if final_role == "lead" and sec_rm in ("lead", "leader"):
            return self._step_like_forager_leader(forager_agent, forager_pos, exclude)
        if final_role == "lead":
            if self._manhattan(self.pos, forager_pos) <= self.cfg.follow_radius:
                if self._lead_parallel_in_band(forager_pos, forager_pos_before, exclude):
                    return True, "move_lead"
                target = self._lead_target_cell(forager_pos, forager_pos_before)
                if self._step_toward(
                    target,
                    exclude=exclude,
                    forbid_cell=forager_pos,
                    velocity_hint=vel,
                ):
                    return True, "move_lead"
                moved = self._orbit_near(forager_pos, self.cfg.follow_radius, exclude=exclude)
                return moved, "move_lead" if moved else "lead_no_step"
            target = self._lead_target_cell(forager_pos, forager_pos_before)
            moved = self._step_toward(
                target,
                exclude=exclude,
                forbid_cell=forager_pos,
                velocity_hint=vel,
            )
            return moved, "move_lead" if moved else "lead_no_step"
        moved = self._orbit_near(forager_pos, self.cfg.follow_radius, exclude=exclude)
        return moved, "move_follow" if moved else "follow_no_step"

    def _update_weights(self):
        if not self.vmove_lead_hist or not self.vmove_follow_hist:
            return
        avg_lead = float(np.mean(self.vmove_lead_hist))
        avg_follow = float(np.mean(self.vmove_follow_hist))
        g = self.cfg.gamma
        if avg_lead > avg_follow:
            self.lead_weight = min(1.0, self.lead_weight + g * (avg_lead - avg_follow))
            self.follow_weight = max(0.0, 1.0 - self.lead_weight)
        elif avg_follow > avg_lead:
            self.follow_weight = min(1.0, self.follow_weight + g * (avg_follow - avg_lead))
            self.lead_weight = max(0.0, 1.0 - self.follow_weight)

    def step(
        self,
        t_in_round: int,
        forager_agent: ForagerAgent,
        round_idx: int,
        global_step: int,
        forager_pos_start: Tuple[int, int],
        forager_pos_before: Optional[Tuple[int, int]],
    ) -> Tuple[str, float]:
        self.revealed_cells.add(tuple(self.pos))
        prev_before_move = tuple(self.pos)
        exclude = self._prev_security_pos

        forager_pos = forager_pos_start
        revealed_count = len(self.revealed_cells)
        total_cells = len(self.env.index)
        stunned_total = sum(1 for x in forager_agent.frames_action if x == "stunned")

        m = self.compute_vtype(forager_pos, revealed_count, total_cells, stunned_total, global_step)
        vtype_t = m["vtype_t"]
        role_by_vtype = "lead" if vtype_t >= self.cfg.vtype_threshold else "follow"

        p_alien_block = self._alien_belief_at(self.pos)
        n_gold_revealed = float(max(0, self._count_revealed_gold_mines()))
        inferred_forager_movement = n_gold_revealed * (1.0 - p_alien_block)
        vmove_core = inferred_forager_movement * p_alien_block * vtype_t

        lead_bonus = self.lead_weight * (1.0 + m["distance_norm_t"])
        follow_bonus = self.follow_weight * (1.0 + (1.0 - m["distance_norm_t"]))
        vmove_lead_t = vmove_core * lead_bonus
        vmove_follow_t = vmove_core * follow_bonus

        self.vmove_lead_hist.append(vmove_lead_t)
        self.vmove_follow_hist.append(vmove_follow_t)
        self._update_weights()

        if vmove_lead_t > vmove_follow_t:
            role_by_choice = "lead"
        elif vmove_follow_t > vmove_lead_t:
            role_by_choice = "follow"
        else:
            role_by_choice = self.cfg.strategy_if_tie

        final_role = role_by_choice if abs(vmove_lead_t - vmove_follow_t) > 1e-9 else role_by_vtype
        role_forced = False
        rm = str(self.cfg.role_mode).lower()
        if rm in ("lead", "leader"):
            final_role, role_forced = "lead", True
        elif rm in ("follow", "follower"):
            final_role, role_forced = "follow", True

        p_belief = self._alien_belief_at(self.pos)
        beat_role_move = self._next_beat_is_role_move

        moved = False
        move_action = "idle"
        rescue_stunned = forager_agent.stunned_steps > 0
        vel_hint = self._forager_velocity(forager_pos, forager_pos_before)
        if rescue_stunned:
            moved = self._step_toward(
                forager_pos,
                exclude=exclude,
                forbid_cell=forager_pos,
                velocity_hint=vel_hint,
            )
            move_action = "move_rescue_stunned" if moved else "rescue_stunned_no_step"
        elif beat_role_move:
            self._next_beat_is_role_move = False
            moved, move_action = self._role_move_one_step(
                forager_agent, forager_pos, forager_pos_before, exclude, final_role
            )
        else:
            self._next_beat_is_role_move = True
            thr = float(self.cfg.chase_belief_threshold)
            if p_belief >= thr:
                moved = False
                move_action = "chase_at_spot"
                self._register_chase_clearance()
            else:
                moved, move_action = self._role_move_one_step(
                    forager_agent, forager_pos, forager_pos_before, exclude, final_role
                )

        if (
            not moved
            and self.pos != forager_pos
            and move_action != "chase_at_spot"
            and move_action != "rescue_stunned_no_step"
        ):
            sec_rm = str(self.cfg.role_mode).lower()
            if sec_rm in ("lead", "leader"):
                m2, act2 = self._step_like_forager_leader(forager_agent, forager_pos, None)
                if m2:
                    move_action = move_action + "+" + act2
                    moved = True
            elif self._step_toward(
                forager_pos,
                exclude=exclude,
                forbid_cell=forager_pos,
                velocity_hint=vel_hint,
            ):
                move_action = move_action + "+fallback_to_forager"
                moved = True

        self.revealed_cells.add(tuple(self.pos))
        p_belief_after = self._alien_belief_at(self.pos)
        vscan_t = self.cfg.e_scan * p_belief_after
        delta_t = float(np.clip(self.lead_weight - self.follow_weight, 0.0, 1.0))
        self._update_escort_trace(prev_before_move, vscan_t)
        self._prev_security_pos = prev_before_move
        self._recent_positions.append(tuple(self.pos))

        self.log.append({
            "global_step": global_step,
            "round": round_idx,
            "step_in_round": t_in_round,
            "security_row": self.pos[0],
            "security_col": self.pos[1],
            "security_move": move_action,
            "security_role": final_role,
            "role_forced": role_forced,
            "rescue_stunned": rescue_stunned,
            "security_beat_role_move": beat_role_move,
            "p_alien_belief": p_belief,
            "chase_belief_threshold": float(self.cfg.chase_belief_threshold),
            "Vscan_t": vscan_t,
            "distance_t": m["distance_t"],
            "distance_norm_t": m["distance_norm_t"],
            "area_revealed_t": m["area_revealed_t"],
            "stun_rate_t": m["stun_rate_t"],
            "Vtype_t": vtype_t,
            "gold_mines_map_count": int(self._count_revealed_gold_mines()),
            "gold_mines_total_ground_truth": int(self._gold_mine_cell_count),
            "inferred_forager_movement": float(inferred_forager_movement),
            "vmove_core": float(vmove_core),
            "Vmove_lead_t": vmove_lead_t,
            "Vmove_follow_t": vmove_follow_t,
            "delta_t": delta_t,
            "lead_weight": self.lead_weight,
            "follow_weight": self.follow_weight,
        })

        return move_action, vscan_t


@dataclass
class AnimFrame:
    turn_idx: int
    phase: str
    step_in_phase: int
    global_step: int
    forager_pos: Tuple[int, int]
    security_pos: Tuple[int, int]
    forager_action: str
    security_move: str
    security_role: str
    vscan: float
    depleted: Set[Tuple[int, int]]
    forager_total_reward: float
    forager_round_reward: float
    revealed_cells: Set[Tuple[int, int]]
    is_start: bool = False


def run_foraging_simulation(
    csv_path: str,
    rounds: int = 3,
    forager_cfg: Optional[ForagerConfig] = None,
    security_cfg: Optional[SecurityConfig] = None,
    seed: int = 0,
    out_prefix: str = "forager",
    save_csv: bool = True,
    save_gif: bool = True,
    forager_identity: str = "auto",
    security_identity: str = "auto",
) -> Dict[str, object]:
    env = load_env_from_csv(csv_path)
    forager_cfg = forager_cfg or ForagerConfig(seed=seed)
    security_cfg = security_cfg or SecurityConfig(seed=seed)

    fi = _normalize_identity(forager_identity)
    si = _normalize_identity(security_identity)
    forager_cfg = replace(forager_cfg, role_mode=fi)
    security_cfg = replace(security_cfg, role_mode=si)

    max_r = int(env.index.get_level_values(0).max())
    max_c = int(env.index.get_level_values(1).max())
    start = (max_r // 2, max_c // 2)
    sec_start: Optional[Tuple[int, int]] = None
    for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
        p = (start[0] + dr, start[1] + dc)
        if p in env.index:
            sec_start = p
            break
    if sec_start is None:
        sec_start = start

    forager = ForagerAgent(env, forager_cfg)
    security = SecurityAgent(env, security_cfg, start_pos=sec_start)
    forager.security_pos = tuple(security.pos)
    forager.security_cleared_alien_cells = security.alien_cells_cleared
    forager.escort_scan_map = security.scan_accum
    forager.escort_trust_mode = fi == "follower" and si == "leader"

    revealed_cells: Set[Tuple[int, int]] = set()
    revealed_cells.add(start)
    revealed_cells.add(sec_start)
    forager.revealed_cells = revealed_cells
    security.revealed_cells = revealed_cells

    frames: List[AnimFrame] = []
    moves = forager_cfg.moves_per_round
    num_turn_rounds = rounds
    global_step = 0

    frames.append(
        AnimFrame(
            turn_idx=-1,
            phase="—",
            step_in_phase=-1,
            global_step=-1,
            forager_pos=tuple(forager.pos),
            security_pos=tuple(security.pos),
            forager_action="(start)",
            security_move="(start)",
            security_role="—",
            vscan=0.0,
            depleted=set(),
            forager_total_reward=forager.total_reward,
            forager_round_reward=forager.round_reward,
            revealed_cells=set(revealed_cells),
            is_start=True,
        )
    )

    for turn_idx in range(num_turn_rounds):
        forager_turn = (turn_idx % 2 == 0)
        forager.current_round = turn_idx

        if forager_turn:
            for k in range(moves):
                if k == 0 and turn_idx > 0:
                    forager.reset_segment_reward()
                forager.security_pos = tuple(security.pos)
                fa = forager.step()
                depleted = {p for p, rem in forager.mine_remaining.items() if rem <= 0}
                frames.append(
                    AnimFrame(
                        turn_idx=turn_idx,
                        phase="forager",
                        step_in_phase=k,
                        global_step=global_step,
                        forager_pos=tuple(forager.pos),
                        security_pos=tuple(security.pos),
                        forager_action=fa,
                        security_move="(frozen)",
                        security_role="—",
                        vscan=0.0,
                        depleted=set(depleted),
                        forager_total_reward=forager.total_reward,
                        forager_round_reward=forager.round_reward,
                        revealed_cells=set(revealed_cells),
                        is_start=False,
                    )
                )
                global_step += 1
        else:
            for k in range(moves):
                forager_pos_start = tuple(forager.pos)
                forager_prev = forager.prev_pos
                security.step(
                    t_in_round=k,
                    forager_agent=forager,
                    round_idx=turn_idx,
                    global_step=global_step,
                    forager_pos_start=forager_pos_start,
                    forager_pos_before=forager_prev,
                )
                forager.escort_travel_dr = security.travel_dr
                forager.escort_travel_dc = security.travel_dc
                sec_row = security.log[-1]
                depleted = {p for p, rem in forager.mine_remaining.items() if rem <= 0}
                frames.append(
                    AnimFrame(
                        turn_idx=turn_idx,
                        phase="security",
                        step_in_phase=k,
                        global_step=global_step,
                        forager_pos=tuple(forager.pos),
                        security_pos=tuple(security.pos),
                        forager_action="(frozen)",
                        security_move=str(sec_row["security_move"]),
                        security_role=str(sec_row["security_role"]),
                        vscan=float(sec_row["Vscan_t"]),
                        depleted=set(depleted),
                        forager_total_reward=forager.total_reward,
                        forager_round_reward=forager.round_reward,
                        revealed_cells=set(revealed_cells),
                        is_start=False,
                    )
                )
                global_step += 1

    forager_log = pd.DataFrame(forager.log)
    security_log = pd.DataFrame(security.log)

    if save_csv:
        forager_log.to_csv(f"{out_prefix}_forager.csv", index=False)
        security_log.to_csv(f"{out_prefix}_security.csv", index=False)

    if save_gif:
        animate_simulation(
            env,
            frames,
            num_turn_rounds=rounds,
            moves_per_round=forager_cfg.moves_per_round,
            outpath=f"{out_prefix}.gif",
        )

    return {
        "env": env,
        "forager": forager,
        "security": security,
        "forager_log": forager_log,
        "security_log": security_log,
        "frames": frames,
        "total_reward": forager.total_reward,
        "rounds": rounds,
        "moves_per_round": forager_cfg.moves_per_round,
        "total_agent_steps": global_step,
        "forager_identity": fi,
        "security_identity": si,
        "revealed_cells": revealed_cells,
    }


def animate_simulation(
    env: pd.DataFrame,
    frames: List[AnimFrame],
    num_turn_rounds: int,
    moves_per_round: int,
    outpath: str = "forager.gif",
):
    env_idx = env.set_index(["Row", "Col"], drop=False)
    max_r, max_c = int(env_idx["Row"].max()), int(env_idx["Col"].max())

    base = np.zeros((max_r + 1, max_c + 1))
    for (r, c), row in env_idx.iterrows():
        base[r, c] = row["reward_prob"]

    fig, ax = plt.subplots(figsize=(7, 6))
    cmap = plt.cm.YlOrRd.copy()
    cmap.set_bad(color="lightgray")
    im = ax.imshow(base, cmap=cmap, origin="upper", vmin=0, vmax=1)
    depleted_cmap = ListedColormap(["#2a2a2a"])
    depleted_layer = np.ma.masked_all((max_r + 1, max_c + 1))
    im_depleted = ax.imshow(
        depleted_layer,
        cmap=depleted_cmap,
        vmin=0,
        vmax=1,
        origin="upper",
        zorder=2,
        interpolation="nearest",
    )

    fd, = ax.plot([], [], "o", color="tab:blue", markersize=11, markeredgecolor="white", markeredgewidth=1.5, label="Forager", zorder=5)
    sd, = ax.plot([], [], "s", color="limegreen", markersize=10, markeredgecolor="black", markeredgewidth=1.2, label="Security", zorder=6)

    ax.set_xticks(range(max_c + 1))
    ax.set_yticks(range(max_r + 1))
    ax.grid(True, linestyle=":", alpha=0.3)
    ax.legend(loc="upper right")

    def _depleted_mine_layer(depleted: Set[Tuple[int, int]]):
        z = np.ma.masked_all((max_r + 1, max_c + 1))
        for (r, c) in depleted:
            if (r, c) not in env_idx.index:
                continue
            if str(env_idx.loc[(r, c), "mine_type"]).strip() == "":
                continue
            z[r, c] = 1.0
        return z

    def init():
        im.set_data(base)
        if frames:
            f0 = frames[0]
            g = base.astype(float).copy()
            rev = f0.revealed_cells
            for r in range(max_r + 1):
                for c in range(max_c + 1):
                    if (r, c) not in env_idx.index:
                        continue
                    if (r, c) not in rev:
                        g[r, c] = np.nan
            im.set_data(g)
            im_depleted.set_data(_depleted_mine_layer(f0.depleted))
            r0, c0 = f0.forager_pos
            rs, cs = f0.security_pos
            if (r0, c0) == (rs, cs):
                r0p, rsp = r0 - 0.22, rs + 0.22
            else:
                r0p, rsp = r0, rs
            fd.set_data([c0], [r0p])
            sd.set_data([cs], [rsp])
        else:
            fd.set_data([], [])
            sd.set_data([], [])
        ax.set_title("")
        return [im, im_depleted, fd, sd]

    def update(i):
        fr = frames[i]
        g = base.astype(float).copy()
        rev = fr.revealed_cells
        for dr, dc in fr.depleted:
            g[dr, dc] = np.nan
        for r in range(max_r + 1):
            for c in range(max_c + 1):
                if (r, c) not in env_idx.index:
                    continue
                if (r, c) not in rev:
                    g[r, c] = np.nan
        im.set_data(g)
        im_depleted.set_data(_depleted_mine_layer(fr.depleted))

        frg, fcg = fr.forager_pos
        sr, sc = fr.security_pos
        if (frg, fcg) == (sr, sc):
            frg_plot, sr_plot = frg - 0.22, sr + 0.22
        else:
            frg_plot, sr_plot = frg, sr
        fd.set_data([fcg], [frg_plot])
        sd.set_data([sc], [sr_plot])

        gstep = "—" if fr.global_step < 0 else str(fr.global_step + 1)
        stext = "—" if fr.step_in_phase < 0 else f"{fr.step_in_phase + 1}/{moves_per_round}"

        if fr.is_start:
            title = (
                f"Alternating rounds: each round = {moves_per_round} steps by ONE agent (no world reset)\n"
                f"INITIAL  |  global step {gstep}  |  next: ROUND 1 = FORAGER x{moves_per_round}\n"
                f"Forager: {fr.forager_action}  |  Security: {fr.security_move}\n"
                f"Reward total={fr.forager_total_reward:.3f}  segment_reward={fr.forager_round_reward:.3f}  |  "
                f"revealed {len(fr.revealed_cells)}/{len(env_idx)} cells"
            )
        else:
            if fr.phase == "forager":
                active = "FORAGER moving"
                sec_extra = ""
            elif fr.phase == "security":
                active = "SECURITY moving"
                sec_extra = f" ({fr.security_role})  |  Vscan={fr.vscan:.3f}"
            else:
                active = str(fr.phase)
                sec_extra = ""
            frozen_note = "other agent frozen" if fr.phase in ("forager", "security") else ""
            title = (
                f"ROUND {fr.turn_idx + 1}/{num_turn_rounds}  |  {active}  |  step {stext}  |  global {gstep}  |  {frozen_note}\n"
                f"Forager: {fr.forager_action}  |  Security: {fr.security_move}{sec_extra}\n"
                f"Reward total={fr.forager_total_reward:.3f}  segment_reward={fr.forager_round_reward:.3f}  |  "
                f"revealed {len(fr.revealed_cells)}/{len(env_idx)} cells"
            )
        ax.set_title(title, fontsize=9)
        return [im, im_depleted, fd, sd]

    anim = FuncAnimation(
        fig,
        update,
        init_func=init,
        frames=len(frames),
        interval=450,
        blit=False,
        repeat=True,
    )
    anim.save(outpath, writer=PillowWriter(fps=3))
    plt.close(fig)


In [31]:
if __name__ == "__main__":
    single_csv = os.path.join(CSV_DIR, "middle_reward_middle_risk_01.csv")
    if not os.path.isfile(single_csv):
        print(f"Skip run: CSV not found: {single_csv}")
    else:
        out = run_foraging_simulation(
            single_csv,
            rounds=30,
            forager_cfg=ForagerConfig(moves_per_round=5, seed=0),
            security_cfg=SecurityConfig(seed=0),
            forager_identity="follower",
            security_identity="leader",
            # Example fixed roles: forager_identity="leader", security_identity="follower"
            out_prefix="forager_sim_sleader",
        )
        print("Total reward:", out["total_reward"])
        print("Frames:", len(out["frames"]))

Total reward: 27.400000000000013
Frames: 151


In [29]:

if __name__ == "__main__":
    single_csv = os.path.join(CSV_DIR, "middle_reward_middle_risk_01.csv")
    if not os.path.isfile(single_csv):
        print(f"Skip run: CSV not found: {single_csv}")
    else:
        out = run_foraging_simulation(
            single_csv,
            rounds=20,
            forager_cfg=ForagerConfig(moves_per_round=5, seed=0),
            security_cfg=SecurityConfig(seed=0),
            forager_identity="leader",
            security_identity="follower",
            # Example fixed roles: forager_identity="leader", security_identity="follower"
            out_prefix="forager_sim_fleader",
        )
        print("Total reward:", out["total_reward"])
        print("Frames:", len(out["frames"]))

Total reward: 19.200000000000006
Frames: 101
